In [10]:
%pip install openai pdfplumber python-docx pillow
%pip install  pdfplumber

from openai import OpenAI


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")
my_api_key[:5], my_api_key[-10:]
client = OpenAI(api_key=my_api_key)

### Extract Information from a PDF
Reads text with pdfplumber and sends to GPT-5-nano for summarization.

In [ ]:
pdf_path = "data/OfferLetter.pdf"  

def extract_text_from_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text.strip()

pdf_text = extract_text_from_pdf(pdf_path)
pdf_text

'EMPLOYMENT OFFER LETTER (CALIFORNIA)\nAcme Corporation\n1234 Market Street\nSan Francisco, CA 94105\nApril 1, 2023\nPrivate & Confidential\nJane Doe\n567 Oak Avenue\nSan Jose, CA 95112\nDear Jane,\nWe are delighted to extend to you an offer of employment with Acme Corporation (“Company”).\nThis letter sets forth the principal terms and conditions of your employment.\n1. Position and Reporting\nYou will join Acme Corporation as a Senior Software Engineer. Your duties will include, but not be\nlimited to: designing, coding, testing, and maintaining enterprise applications; collaborating with\ncross-functional teams; mentoring junior engineers; and ensuring compliance with internal security\nand data privacy standards. You will report directly to John Smith, Chief Technology Officer. This\nis a full-time, exempt position under California wage and hour law. Your primary work location will\nbe San Francisco, CA, but you may be required to travel up to 10% of your time.\n2. Start Date and D

In [16]:

prompt = f"Extract and summarize key details from this PDF:\n\n{pdf_text}"

#Open AI API call to summarize and extract key details from the DOCX text
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
          {"role": "system", "content": '''You summarize and extract data from PDF text. 
         Flag any items for compliance and risk
         If the number of hours of work per week is greater than 40, flag for overtime risk.
         If the compensation is less than $15 per hour, flag for minimum wage risk.
         If the contract duration is less than 3 months, flag for short-term contract risk.
         If the contract does not include a non-compete clause, flag for competitive risk.
         If the contract does not include a confidentiality clause, flag for data security risk.
         If the contract does not include a termination clause, flag for termination risk.
         If the contract does not include a dispute resolution clause, flag for legal risk.
         If the contract does not include a governing law clause, flag for jurisdictional risk.
         If the contract does not include an indemnification clause, flag for liability risk.'''},
        {"role": "user", "content": prompt}
    ]

)


In [18]:


print("Extracted Info from PDF:\n")
print(response.choices[0].message.content)

Extracted Info from PDF:

Here is a concise extraction and risk/compliance review of the Employment Offer Letter.

Key details
- Parties and date
  - Employer: Acme Corporation, 1234 Market Street, San Francisco, CA 94105
  - Employee: Jane Doe, 567 Oak Avenue, San Jose, CA 95112
  - Date of offer: April 1, 2023
  - Confidentiality: Private & Confidential

- Position and reporting
  - Title: Senior Software Engineer
  - Duties: designing, coding, testing, maintaining enterprise apps; cross-functional collaboration; mentoring; security/data privacy compliance
  - Reports to: John Smith, Chief Technology Officer
  - Status: Full-time, exempt under California wage and hour law
  - Primary work location: San Francisco, CA
  - Travel: Up to 10% of time

- Start date and duration
  - Start date: April 24, 2023 (contingent on listed conditions)
  - Offer expiration: April 15, 2023 (acceptance must be signed and returned by that date)

- Compensation
  - Base salary: $150,000 per year, paid bi

In [20]:
import ollama
#Open AI API call to summarize and extract key details from the PDF text
prompt = f"Extract and summarize key details from this PDF:\n\n{pdf_text}"

response = ollama.chat(
    model="llama3",
    messages=[
        {"role": "system", "content": '''You summarize and extract data from PDF text. 
        Flag any items for compliance and risk
         If the number of hours of work per week is greater than 40, flag for overtime risk.
         If the compensation is less than $15 per hour, flag for minimum wage risk.
         If the contract duration is less than 3 months, flag for short-term contract risk.
         If the contract does not include a non-compete clause, flag for competitive risk.
         If the contract does not include a confidentiality clause, flag for data security risk.
         If the contract does not include a termination clause, flag for termination risk.
         If the contract does not include a dispute resolution clause, flag for legal risk.
         If the contract does not include a governing law clause, flag for jurisdictional risk.
         If the contract does not include an indemnification clause, flag for liability risk.'''},
        {"role": "user", "content": prompt}
    ]
)

print("Extracted Info from PDF:\n")
print(response['message']['content'])


Extracted Info from PDF:

Here are the key details extracted from the PDF:

**Employment Offer**

* Job Title: Senior Software Engineer
* Reporting Manager: John Smith, Chief Technology Officer
* Primary Work Location: San Francisco, CA (with occasional travel up to 10% of the time)
* Full-time, exempt position under California wage and hour law

**Compensation**

* Annual Base Salary: $150,000 (bi-weekly installments)
* Bonus: Eligible for a discretionary annual bonus targeted at 10% of base salary
* Equity Grant: Restricted Stock Units (RSUs) with 10,000 units vesting over 4 years with a 1-year cliff (subject to Board approval)

**Benefits and Paid Time Off**

* Company benefits as described in the Employee Handbook:
	+ Health, dental, vision insurance
	+ 401(k) with 4% match
	+ 15 vacation days per year
	+ 10 paid holidays
	+ Sick leave in accordance with California Healthy Workplaces, Healthy Families Act

**Confidentiality and IP**

* Required to sign the Confidential Information 


### Extract Information from a DOCX (Word) File
Reads paragraphs and asks GPT-5-nano for key points.

In [ ]:

docx_path = "data/Project_List.docx" 

def extract_text_from_docx(path):
    document = docx.Document(path)
    return "\n".join([p.text for p in document.paragraphs])

docx_text = extract_text_from_docx(docx_path)

print (f"Extracted DOCX Text (first 500 chars):\n{docx_text[:500]}\n")
prompt = f"Extract key points from this DOCX content:\n\n{docx_text[:8000]}"

In [21]:
#Open AI API call to summarize and extract key details from the DOCX text
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": '''You summarize and extract structured data from Word documents. Also, extract 
         a list of skills used in that project. Reurn as JSON with the following format:
            {
            "projects": [
                {
                "name": "Project Name",
                "description": "Brief description of the project",
                "skills": ["Skill1", "Skill2", "Skill3"]
                },
                ...
            ]
            }
        '''},
        {"role": "user", "content": prompt}
    ]
)

print("Extracted Info from DOCX:\n")
print(response.choices[0].message.content)


Extracted Info from DOCX:

{
  "projects": [
    {
      "name": "Acme Offer Letter - Senior Software Engineer (Jane Doe)",
      "description": "California employment offer letter from Acme Corporation for Jane Doe. Core terms include: position as Senior Software Engineer reporting to the CTO; full-time exempt status; primary location in San Francisco with up to 10% travel; anticipated start date April 24, 2023; offer expiration April 15, 2023. Compensation comprises a $150,000 base salary (bi-weekly pay), discretionary annual bonus targeting 10% of base salary, and a grant of 10,000 Restricted Stock Units (RSUs) vesting over 4 years with a 1-year cliff (subject to board approval). Benefits include health/dental/vision, 401(k) with 4% match, 15 vacation days, 10 holidays, and sick leave per California law. Key agreements include Confidential Information and Invention Assignment Agreement (CIIAA) and Mutual Arbitration Agreement; contingencies cover verification of work authorization, 

In [22]:
#Ollama Extraction for DOCX text
import ollama

response = ollama.chat (
    model="llama3",
    messages=[
        {"role": "system", "content": "You summarize and extract structured data from Word documents."},
        {"role": "user", "content": prompt}
    ]
)

print("Extracted Info from DOCX:\n")
print(response["message"]["content"])


Extracted Info from DOCX:

Here are the key details extracted from the Employment Offer Letter:

**Job Details**

* Position: Senior Software Engineer
* Reporting: John Smith, Chief Technology Officer
* Primary work location: San Francisco, CA
* Travel requirement: Up to 10% of your time

**Compensation**

* Base salary: $150,000 per year (bi-weekly installments)
* Bonus: Discretionary annual bonus targeted at 10% of base salary
* Equity grant: 10,000 Restricted Stock Units (RSUs), vesting over 4 years with a 1-year cliff

**Benefits and Paid Time Off**

* Benefits:
	+ Health
	+ Dental
	+ Vision
	+ 401(k) with 4% match
* Paid time off:
	+ 15 vacation days per year
	+ 10 paid holidays
	+ Sick leave in accordance with California Healthy Workplaces, Healthy Families Act

**Confidentiality and Intellectual Property**

* Confidential Information and Invention Assignment Agreement (CIIAA): Must be signed
* Key obligations: Protect proprietary information, assign inventions to the company, av